In [2]:
import numpy as np
from scipy.optimize import brentq

In [3]:
def find_roots_in_range(serch_range, get_beta,M,lam,q,s,t,granularity,f_roots):
    def f(alpha):
            beta=get_beta(alpha)
            term1=M * (beta-1) * t**2
            term2=lam * q * alpha**q * beta**(q-1) * (s * t)**q
            return term1+term2

    target_alpha=None
    r_min,r_max=serch_range[0],serch_range[1]
    probes = np.linspace(r_min, r_max, granularity) # split the search_range to probes
    for i in range(len(probes) - 1):
        f1=f(probes[i]) # get the f of boundary of probe
        f2=f(probes[i+1]) # get the f of boundary of probe
        if f1*f2>=0: # if boundaries are same sign, skip
            continue
        else:
            target_alpha = brentq(f, probes[i], probes[i+1]) # Brent's Method finding alpha
            target_beta = get_beta(target_alpha) # get beta
            f_roots.append((target_alpha,target_beta),) # storing result
    return f_roots

In [4]:
def range_solve(M, lam, q, s, t,granularity):
    def objective(alpha, beta):
        return (M/2) * ((alpha-1)**2 * s**2 + (beta-1)**2 * t**2) + lam * (alpha*beta*s*t)**q
    
    dic={(0,0):(M/2) * (s**2 + t**2),
            (0,1):(M/2) * (s**2),
            (1,0):(M/2) * (t**2),
            (1,1):lam*((s*t)**q)} # find the objective value of boundaries
    
    best_alpha, best_beta = min(dic, key=dic.get) # get the best_alpha, best_beta of min objective value
    min_obj = dic[(best_alpha, best_beta)] # get min objective value

    swapped = False
    if s < t: # check if we are solve alpha or beta
        s, t = t, s
        swapped = True
        print("here is finding beta")
    
    def get_beta1(alpha): # + quadratic formula
        inner = 1 - 4 * alpha * (1 - alpha) * (s**2 / t**2)
        inner=np.maximum(0, inner)
        return (1 + np.sqrt(inner)) / 2 
    
    def get_beta2(alpha): # - quadratic formula
        inner = 1 - 4 * alpha * (1 - alpha) * (s**2 / t**2)
        inner=np.maximum(0, inner)
        return (1 - np.sqrt(inner)) / 2
    
    # get the domain
    bound_inner = 1 - (t / s)**2
    bound_low = (1 - np.sqrt(bound_inner)) / 2
    bound_high = (1 + np.sqrt(bound_inner)) / 2
    search_ranges = [(1e-9, bound_low), (bound_high, 1 - 1e-9)]

    f_roots=[]
    num_root=0
    for r in search_ranges:
        print("--------------")
        print(f"search_ranges={r}")

        f_roots=find_roots_in_range(r, get_beta1, M,lam,q,s,t,granularity,f_roots) # find root using beta1 formula
        if len(f_roots)==num_root: # if no new root generated
            print("beta1 no root")
        else:
            print("root number + 1")
            num_root+=1
        f_roots=find_roots_in_range(r, get_beta2, M,lam,q,s,t,granularity,f_roots) # find root using beta1 formula
        if len(f_roots)==num_root: # if no new root generated
            print("beta2 no root")
        else:
            print("root number + 1")
            num_root+=1
    if len(f_roots)==0: # if no any root, stop
        print("--------------")
        print("no root")
    else:
        print("--------------")
        print(f"root number = {len(f_roots)}")
        if swapped:
            f_roots = [(y, x) for x, y in f_roots]
        for i in f_roots: # check if existing any root with smaller objective value than the boundary, and get the best root
            target_alpha, target_beta=i[0],i[1]
            current_obj = objective(target_alpha, target_beta)
            if current_obj < min_obj:
                min_obj = current_obj
                best_alpha, best_beta = target_alpha, target_beta

        print("--------------")
        print(f"roots={list(f_roots)}")
        print(f"best_alpha={best_alpha:.8f}, best_beta={best_beta:.8f}")



In [5]:
def direct_solve(M, lam, q, s, t,granularity):
    def get_beta(alpha):
        inner = (-M * (alpha-1) * s**2)/(lam * q * (s * t)**q * alpha **(q-1))
        beta = inner**(1/q)
        return beta
    
    def objective(alpha, beta):
        return (M/2) * ((alpha-1)**2 * s**2 + (beta-1)**2 * t**2) + lam * (alpha*beta*s*t)**q
    
    dic={(0,0):(M/2) * (s**2 + t**2),
            (0,1):(M/2) * (s**2),
            (1,0):(M/2) * (t**2),
            (1,1):lam*((s*t)**q)} # find the objective value of boundaries
    
    best_alpha, best_beta = min(dic, key=dic.get)
    min_obj = dic[(best_alpha, best_beta)]
    
    r=[1e-12, 1 - 1e-12]
    f_roots=[]
    f_roots=find_roots_in_range(r, get_beta, M,lam,q,s,t,granularity,f_roots)
    if len(f_roots)==0: # if no any root, stop
        print("--------------")
        print("no root")
    else:
        print("--------------")
        print(f"root number = {len(f_roots)}")
        for i in f_roots: # check if existing any root with smaller objective value than the boundary, and get the best root
            target_alpha, target_beta=i[0],i[1]
            current_obj = objective(target_alpha, target_beta)
            if current_obj < min_obj:
                min_obj = current_obj
                best_alpha, best_beta = target_alpha, target_beta
        print("--------------")
        print(f"roots={list(f_roots)}")
        print(f"best_alpha={best_alpha:.8f}, best_beta={best_beta:.8f}")

When the granularity is not small enough, sometimes there are 3 roots in range_solve, but only 1 root in direct_solve.

It‘s possible that no roots are found within any sub-ranges or via either beta formula

In [100]:
'''
Repeatly run this cell to get different random results.
'''
M, lam, q = 1, 1e-2, 0.5
num=0

s = 10**np.random.uniform(-2, 1)
t = 10**np.random.uniform(-2, 1)
s,t=1.1997106075286865,0.1737809181213379
print(f"s,t={s,t}")
print("")
print("range_solve")
range_solve(M, lam, q, s, t,1000)#1000 is the granularity
print("")
print("=============================")
print("")
print("direc_solve")
direct_solve(M, lam, q, s, t,2000)

s,t=(1.1997106075286865, 0.1737809181213379)

range_solve
--------------
search_ranges=(1e-09, 0.005273360197350696)
root number + 1
beta2 no root
--------------
search_ranges=(0.9947266398026493, 0.999999999)
root number + 1
root number + 1
--------------
root number = 3
--------------
roots=[(2.515722333430362e-06, 0.9998800880361232), (0.9984763402247951, 0.9213002377774342), (0.9998793911129816, 0.005780875775806438)]
best_alpha=0.99847634, best_beta=0.92130024


direc_solve
--------------
root number = 3
--------------
roots=[(2.515722342119495e-06, 0.9998800930480929), (0.9984763402247467, 0.9213002377745914), (0.9998793911124879, 0.005780875823149548)]
best_alpha=0.99847634, best_beta=0.92130024
